# UC-SEARCH-1 — Trigger Full Catalog Reindex (Obfuscated ES)

**Persona:** Platform Administrator

**Goal:** Verify no active reindex is running, trigger a full obfuscated
Elasticsearch reindex from the PostgreSQL source, monitor progress via the Logs
API, then spot-check a known GeoID to confirm the item is searchable in ES.

**Prerequisites:**
- ES cluster healthy and reachable from the platform
- `elasticsearch_obfuscated` driver configured for the target catalog
  (commit 8c9c654 — obfuscated ES driver DONE)
- Admin JWT in `DYNASTORE_ADMIN_TOKEN`
- A known GeoID in `SAMPLE_GEOID` (e.g. `AFR_ETH_ET_AM`)

**Env vars required:** `DYNASTORE_BASE_URL`, `DYNASTORE_ADMIN_TOKEN`,
`CATALOG_ID`, `SAMPLE_GEOID`

**Key routes:**
- `GET /logs/catalogs/{id}` — catalog log stream
- `POST /search/catalogs/{id}/reindex` — trigger reindex (202 Accepted)
- `GET /search/catalogs/{id}/geoid/{geoid}` — spot-check via GeoID lookup

**Memory refs:**
- `project_geoid_obfuscated_es_rework` — DONE commit 7ca5408: per-tenant ES + simplify_to_fit
- `project_geoid_logs_es_backend` — DONE f4ff70d: ES log backend, Kibana dashboard

In [ ]:
import os
import json
import time

import httpx
from dotenv import load_dotenv

load_dotenv()

BASE_URL     = os.environ["DYNASTORE_BASE_URL"]
ADMIN_TOKEN  = os.environ["DYNASTORE_ADMIN_TOKEN"]
CATALOG_ID   = os.environ["CATALOG_ID"]
SAMPLE_GEOID = os.environ.get("SAMPLE_GEOID", "AFR_ETH_ET_AM")

headers = {
    "Authorization": f"Bearer {ADMIN_TOKEN}",
    "Content-Type": "application/json",
}
client = httpx.Client(base_url=BASE_URL, headers=headers, timeout=60.0)
print(f"Connected to {BASE_URL}")
print(f"Catalog: {CATALOG_ID}  Sample GeoID: {SAMPLE_GEOID}")

In [ ]:
# Pre-flight: check recent reindex logs — ensure no active job is running
r = client.get(
    f"/logs/catalogs/{CATALOG_ID}",
    params={
        "event_type": "REINDEX",
        "level": "INFO",
        "limit": 5,
    },
)
print(r.status_code, r.text[:600])
assert r.status_code == 200, f"Expected 200, got {r.status_code}: {r.text}"

logs_resp = r.json()
recent_logs = logs_resp.get("logs", [])
print(f"\nRecent REINDEX log entries: {len(recent_logs)}")
for entry in recent_logs:
    print(f"  [{entry.get('level'):7s}] {entry.get('timestamp', 'n/a')}  {entry.get('message', '')[:80]}")

# Heuristic: if the last entry does not contain 'completed' or 'failed',
# a reindex may still be in progress.  Confirm before proceeding.
active_messages = [
    e for e in recent_logs
    if "started" in str(e.get("message", "")).lower()
    and "completed" not in str(e.get("message", "")).lower()
]
if active_messages:
    print("\nWARN: A recent REINDEX job may still be active. Proceeding anyway.")
else:
    print("\nNo active reindex detected — safe to proceed.")

In [ ]:
# Trigger full catalog reindex via query params (mode and driver)
# POST /search/catalogs/{catalog_id}/reindex
#   mode=obfuscated   → build the obfuscated GeoID index
#   driver=elasticsearch_obfuscated → select the ES obfuscated driver explicitly

r = client.post(
    f"/search/catalogs/{CATALOG_ID}/reindex",
    params={
        "mode": "obfuscated",
        "driver": "elasticsearch_obfuscated",
    },
)
print(r.status_code, r.text[:400])
assert r.status_code == 202, f"Expected 202 Accepted, got {r.status_code}: {r.text}"

reindex_resp = r.json()
job_id = reindex_resp.get("job_id") or reindex_resp.get("task_id")
print(f"\nReindex accepted  job_id={job_id}")
print(json.dumps(reindex_resp, indent=2))

In [ ]:
# Poll logs for reindex completion
# The reindex runs asynchronously; completion is reported as a REINDEX log entry
# with message containing 'completed' at INFO level.

MAX_POLL = 5
POLL_SLEEP = 5  # seconds
completed = False

for attempt in range(1, MAX_POLL + 1):
    time.sleep(POLL_SLEEP)
    r = client.get(
        f"/logs/catalogs/{CATALOG_ID}",
        params={"event_type": "REINDEX", "limit": 20},
    )
    assert r.status_code == 200, f"Log poll failed: {r.status_code}: {r.text}"
    entries = r.json().get("logs", [])
    print(f"\n[poll {attempt}/{MAX_POLL}] REINDEX log entries: {len(entries)}")
    for e in entries[:5]:
        print(f"  [{e.get('level'):7s}] {e.get('message', '')[:100]}")
    if any("completed" in str(e.get("message", "")).lower() for e in entries):
        completed = True
        print("\nReindex COMPLETED detected in logs.")
        break

if not completed:
    print(
        f"\nReindex did not complete within {MAX_POLL * POLL_SLEEP}s. "
        "Check Kibana or the logs endpoint for progress."
    )

In [ ]:
# Spot-check: verify the sample GeoID is searchable after reindex
# GET /search/catalogs/{catalog_id}/geoid/{geoid}

r = client.get(f"/search/catalogs/{CATALOG_ID}/geoid/{SAMPLE_GEOID}")
print(r.status_code, r.text[:600])
assert r.status_code == 200, (
    f"Expected 200 for GeoID '{SAMPLE_GEOID}', got {r.status_code}: {r.text}"
)

geoid_result = r.json()
matched = geoid_result.get("features", geoid_result.get("items", []))
print(f"\nGeoID '{SAMPLE_GEOID}' matched {len(matched)} item(s) in ES index")
for item in matched[:3]:
    print(f"  id={item.get('id')}  collection={item.get('collection')}")

## Edge cases

### ES cluster unreachable

If the Elasticsearch cluster is down at reindex time, the `elasticsearch_obfuscated`
driver logs a `WARNING`-level `REINDEX` event and returns without raising.  PostgreSQL
reads (`/features`, `/stac`) continue uninterrupted because ES is a secondary index only.

In [ ]:
# ES cluster unreachable — expected WARNING in logs, non-fatal
#
# In this scenario the reindex 202 response still arrives (the task is accepted)
# but the ES bulk indexing fails.  Check logs for WARNING events:
#
warn_r = client.get(
    f"/logs/catalogs/{CATALOG_ID}",
    params={"event_type": "REINDEX", "level": "WARNING", "limit": 10},
)
assert warn_r.status_code == 200, f"Expected 200, got {warn_r.status_code}: {warn_r.text}"
warn_entries = warn_r.json().get("logs", [])
print(f"REINDEX WARNING log entries: {len(warn_entries)}")
for e in warn_entries[:3]:
    print(f"  {e.get('message', '')[:120]}")
print()
print("If ES is unreachable: logs show WARNING; PG reads (OGC Features / STAC) remain available.")

In [ ]:
# Legacy obfuscated index without geometry field
#
# Older ES indices created before the obfuscated driver update (commit 8c9c654)
# may lack the `geometry` field mapping.  When the driver detects this, it calls
# _ensure_obfuscated_index() which:
#   1. Drops the existing index (DELETE /{catalog_id}_obfuscated)
#   2. Re-creates it with the correct geo_shape mapping
#   3. Re-triggers bulk indexing of all items
#
# This is a one-time destructive migration; the window where the index is absent
# means GeoID lookups return 0 results until the reindex completes.
#
# Mitigation: run this during a maintenance window, or keep PG as the primary
# search backend until the new index is ready.

print("Legacy index migration: _ensure_obfuscated_index drops and re-creates the ES index.")
print("  → geometry field added to mapping")
print("  → all items re-indexed in bulk")
print("  → GeoID lookups return 0 results during migration window (use PG fallback)")

In [ ]:
# Cross-check PG vs ES item count
# Compare total items from the OGC Features endpoint (PostgreSQL-backed)
# against the number returned by a broad ES search to detect index drift.

# PG count via OGC Features collection items (numberMatched)
pg_r = client.get(
    f"/features/catalogs/{CATALOG_ID}/collections",
    params={"limit": 1},
)
if pg_r.status_code == 200:
    collections_resp = pg_r.json()
    collections = collections_resp.get("collections", [])
    print(f"PG collections available: {len(collections)}")
else:
    print(f"Features endpoint unavailable ({pg_r.status_code}) — skipping PG/ES count comparison.")

# ES count via geoid batch search (all known geoids)
es_r = client.post(
    f"/search/catalogs/{CATALOG_ID}/geoid",
    content=json.dumps({"geoids": [SAMPLE_GEOID], "limit": 1}),
)
if es_r.status_code == 200:
    es_result = es_r.json()
    es_total = es_result.get("numberMatched") or es_result.get("total")
    print(f"ES total for sample GeoID '{SAMPLE_GEOID}': {es_total}")
else:
    print(f"ES search unavailable ({es_r.status_code}) — index may still be building.")

client.close()